# Quickstart

In [ ]:
from sqlalchemy import text
import pandas as pd

from pyquant import config, db, x_client, openai_client, yfinance_client, analyze, price, signals

In [ ]:
engine = db.get_engine(config.settings.DB_URL)
db.create_tables(engine)
conn = engine.connect()

In [ ]:
tweets = x_client.get_tweets(['alice','bob'], None, None)
if not config.settings.HAS_X:
    print('Mock mode: using sample tweets')
df_tweets = pd.DataFrame(tweets)
display(df_tweets)
for t in tweets:
    conn.execute(text("INSERT OR IGNORE INTO tweets (user_handle, tweet_id, text, created_at) VALUES (:user_handle, :tweet_id, :text, :created_at)"), t)

In [ ]:
analyses = openai_client.analyze_tweets(tweets)
rows = []
for a in analyses:
    for coin in a['coins']:
        t0 = pd.to_datetime(next(t['created_at'] for t in tweets if t['tweet_id']==a['tweet_id']))
        pw = yfinance_client.get_price_window(coin, t0, 7)
        if not pw:
            continue
        conn.execute(text("INSERT INTO tweet_analysis (tweet_id, coin_symbol, stance, rationale, confidence, model, created_at) VALUES (:tweet_id, :coin_symbol, :stance, :rationale, :confidence, :model, CURRENT_TIMESTAMP)"), {**a, 'coin_symbol': coin})
        conn.execute(text("INSERT INTO price_window (coin_symbol, t0, t1, max_price, min_price, vol) VALUES (:coin_symbol, :t0, :t1, :max_price, :min_price, :vol)"), {**pw, 'coin_symbol': coin})
        mom = price.momentum(pw['t0_price'], pw['max_price'], pw['min_price'])
        score = signals.score_signal(a['confidence'], analyze.stance_to_dir(a['stance']), mom)
        conn.execute(text("INSERT INTO signals (tweet_id, coin_symbol, signal_score, horizon_days, comment, created_at) VALUES (:tweet_id, :coin_symbol, :signal_score, 7, '', CURRENT_TIMESTAMP)"), {'tweet_id': a['tweet_id'], 'coin_symbol': coin, 'signal_score': score})
        rows.append({'tweet_id': a['tweet_id'], 'coin_symbol': coin, 'signal_score': score})
pd.DataFrame(rows).sort_values('signal_score', ascending=False)

In [ ]:
def run_pipeline(handles, since=None, until=None, horizon_days=7):
    engine = db.get_engine(config.settings.DB_URL)
    db.create_tables(engine)
    conn = engine.connect()
    tweets = x_client.get_tweets(handles, since, until)
    for t in tweets:
        conn.execute(text("INSERT OR IGNORE INTO tweets (user_handle, tweet_id, text, created_at) VALUES (:user_handle, :tweet_id, :text, :created_at)"), t)
    analyses = openai_client.analyze_tweets(tweets)
    rows = []
    for a in analyses:
        for coin in a['coins']:
            t0 = pd.to_datetime(next(t['created_at'] for t in tweets if t['tweet_id']==a['tweet_id']))
            pw = yfinance_client.get_price_window(coin, t0, horizon_days)
            if not pw:
                continue
            conn.execute(text("INSERT INTO tweet_analysis (tweet_id, coin_symbol, stance, rationale, confidence, model, created_at) VALUES (:tweet_id, :coin_symbol, :stance, :rationale, :confidence, :model, CURRENT_TIMESTAMP)"), {**a, 'coin_symbol': coin})
            conn.execute(text("INSERT INTO price_window (coin_symbol, t0, t1, max_price, min_price, vol) VALUES (:coin_symbol, :t0, :t1, :max_price, :min_price, :vol)"), {**pw, 'coin_symbol': coin})
            mom = price.momentum(pw['t0_price'], pw['max_price'], pw['min_price'])
            score = signals.score_signal(a['confidence'], analyze.stance_to_dir(a['stance']), mom)
            conn.execute(text("INSERT INTO signals (tweet_id, coin_symbol, signal_score, horizon_days, comment, created_at) VALUES (:tweet_id, :coin_symbol, :signal_score, :horizon_days, '', CURRENT_TIMESTAMP)"), {'tweet_id': a['tweet_id'], 'coin_symbol': coin, 'signal_score': score, 'horizon_days': horizon_days})
            rows.append({'tweet_id': a['tweet_id'], 'coin_symbol': coin, 'signal_score': score})
    return pd.DataFrame(rows).sort_values('signal_score', ascending=False)

run_pipeline(['alice','bob'])